# Two-Tower Neural Retrieval: Evidence and Trade-offs

The objective-conditioned retriever adds **1.029 percentage points** to the frozen baseline's candidate-recall ceiling on Fold 0 (95% paired interval: **0.931–1.120 pp**). This is complementary retrieval evidence, not a ranked recommendation score. Subsequent [ANN benchmarking](06_ann_benchmark.ipynb) and the separate [100-candidate baseline ranker](08_ranking_evaluation.ipynb) are complete. Independently selected neural-source ranking remains unmeasured.

This notebook reads compact committed results. It covers training, durable recovery, exact catalogue export, and an independently audited comparison. CI executes real Jupyter kernels; figures and tables are embedded for GitHub readers. Production computation lives in typed modules and CLI workflows.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "reports" / "metrics" / "two_tower_resume_proof.json").is_file()
)
resume_path = ROOT / 'reports/metrics/two_tower_resume_proof.json'
fold_path = ROOT / 'reports/metrics/two_tower_fold0_training.json'

resume = json.loads(resume_path.read_text(encoding='utf-8'))
fold = json.loads(fold_path.read_text(encoding='utf-8')) if fold_path.exists() else None
print(f"resume_proof={resume['status']} commit={resume['code_commit'][:7]}")
print(f"fold0_training={'available' if fold is not None else 'pending'}")

## Proven cross-worker durability

In [ ]:
proof_rows = [
    ('Durable checkpoint objects', resume['checkpoint_objects']),
    ('Checkpoint bytes', f"{resume['checkpoint_bytes']:,}"),
    ('Resumed from step', resume['resumed_from_step']),
    ('Final step', resume['final_step']),
    ('Advanced steps', resume['advanced_steps']),
    ('Pipeline elapsed seconds', resume['pipeline_elapsed_seconds']),
]
pd.DataFrame(proof_rows, columns=['Invariant', 'Measured result'])

The resume proof uses a fresh managed worker for the continuation stage. The final global step must strictly exceed the restored step; checkpoint presence alone is not counted as a pass.

## Fold 0 experimental contract

In [ ]:
contract = pd.DataFrame([
    ('Held-out fold', 0),
    ('Training folds', '1, 2, 3, 4'),
    ('Candidate depths', '20 / 50 / 100 / 200 / 400 / 800'),
    ('Primary evidence', 'incremental Recall@20 ceiling vs frozen base retrieval'),
    ('Complementarity evidence', 'neural-only positive hits by objective'),
    ('Uncertainty', 'paired session-level percentile bootstrap'),
    ('Scaling rule', 'do not run folds 1–4 unless Fold 0 adds held-out value'),
], columns=['Contract', 'Value'])
contract


## Fold 0 training result

In [ ]:
if fold is None:
    print('Fold 0 full training has not been published yet. Run the managed fold workflow, then rerun this notebook.')
else:
    display(pd.DataFrame([
        ('status', fold['status']),
        ('global_step', fold['global_step']),
        ('completed_epochs', fold['completed_epochs']),
        ('best_valid_loss', fold['best_valid_loss']),
        ('billable_seconds', fold['billable_seconds']),
        ('checkpoint_bytes', fold['checkpoint_bytes']),
    ], columns=['Metric', 'Value']))


## Learning dynamics
The managed run completed four epochs and 9,600 optimizer steps. The minimum validation loss occurred at epoch 1; the evaluator uses that saved best checkpoint. Loss and MRR below are diagnostics against sampled negatives. They do not measure full-catalogue Recall@20.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = fold['history']
epochs = np.array([row['epoch'] + 1 for row in history])
train_loss = np.array([row['train']['loss'] for row in history])
valid_loss = np.array([row['valid']['loss'] for row in history])
best = int(np.argmin(valid_loss))
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), facecolor='#f7f9fc')
for ax in axes:
    ax.set_facecolor('#f7f9fc')
    ax.set_xticks(epochs)
    ax.set_xlabel('Completed epoch')
    ax.grid(axis='y', color='#dce3ec', linewidth=0.8)
    ax.set_axisbelow(True)
axes[0].plot(epochs, train_loss, 'o-', color='#3366b0', lw=2.3, label='Training')
axes[0].plot(epochs, valid_loss, 'o-', color='#c05b43', lw=2.3, label='Validation')
axes[0].scatter(epochs[best], valid_loss[best], s=140, facecolors='none',
                edgecolors='#17324f', linewidths=2, zorder=5)
axes[0].annotate('Selected checkpoint', xy=(epochs[best], valid_loss[best]),
                 xytext=(1.25, 4.35), color='#17324f', fontsize=10)
axes[0].set_title('Loss diverges after the first epoch', loc='left', fontweight='bold')
axes[0].set_ylabel('Training objective loss')
axes[0].legend(frameon=False)
axes[1].plot(epochs, [row['train']['mrr'] for row in history], 'o-', color='#3366b0', lw=2.3)
axes[1].plot(epochs, [row['valid']['mrr'] for row in history], 'o-', color='#c05b43', lw=2.3)
axes[1].set_title('Validation MRR plateaus', loc='left', fontweight='bold')
axes[1].set_ylabel('MRR against sampled negatives')
axes[1].set_ylim(0.18, 0.57)
fig.suptitle('OTTO | Two-tower Fold 0 training', x=0.07, ha='left',
             fontsize=19, fontweight='bold', color='#17324f')
fig.text(0.07, 0.015, '4 epochs · 9,600 steps · 324.4 s training · 621 s billable instance time\n'
         'Source: committed SageMaker training manifest. Paired retrieval quality results are reported below.',
         fontsize=9, color='#536174')
fig.subplots_adjust(left=0.07, right=0.98, top=0.79, bottom=0.23, wspace=0.28)
figure_path = ROOT / 'reports/figures/two_tower_learning_curves.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=180, facecolor=fig.get_facecolor())
plt.show()


Validation loss rises while training loss falls, consistent with overfitting on this split. The best checkpoint is used for the retrieval comparison below. Fold 0 participates in checkpoint selection, so its results are exploratory validation. An untouched evaluation remains necessary for a generalization claim.

## Full-catalogue export completed

The saved Fold 0 checkpoint produced the top 800 candidates for every held-out session and objective. AWS confirmed completion; all 96 prediction files and their checksum receipts are in S3. The chart measures export coverage and runtime, including retrieval and serialization. These batch timings do not measure single-request serving latency.

In [ ]:
export = json.loads((ROOT / 'reports/metrics/two_tower_fold0_export.json').read_text())
assert export['status'] == 'passed'
assert all(row['sessions'] == export['sessions'] for row in export['objectives'].values())
pd.DataFrame([
    ('Held-out sessions', f"{export['sessions']:,}"),
    ('Catalogue items', f"{export['catalogue_items']:,}"),
    ('Candidates per session / objective', export['search']['k']),
    ('Verified uploaded prediction parts', export['prediction_parts']),
    ('Export elapsed seconds', round(export['export_seconds'], 3)),
    ('AWS billable instance seconds', export['billable_seconds']),
    ('Instance type', export['instance_type']),
    ('Quality comparison', 'Completed; paired analysis below'),
], columns=['Export evidence', 'Observed'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), facecolor='#f7f9fc')
ax.set_facecolor('#f7f9fc')
colors = {'clicks': '#3366b0', 'carts': '#168777', 'orders': '#b77a2b'}
for objective, color in colors.items():
    durations = export['objectives'][objective]['bucket_seconds']
    ax.plot(range(len(durations)), durations, 'o-', markersize=3.5,
            linewidth=1.6, color=color, label=objective.title())
ax.set_ylim(0, 4)
ax.set_xlim(-0.5, 31.5)
ax.set_xticks([0, 4, 8, 12, 16, 20, 24, 28, 31])
ax.set_xlabel('Held-out partition')
ax.set_ylabel('Retrieval + serialization time (seconds)')
ax.grid(axis='y', color='#dce3ec', linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, ncols=3, loc='lower right')
fig.suptitle('OTTO | Full-catalogue export completed', x=0.085, ha='left',
             fontsize=19, fontweight='bold', color='#17324f')
fig.text(0.085, 0.865,
         f"{export['sessions']:,} sessions   /   {export['catalogue_items']:,} catalogue items   /   96 saved prediction parts",
         fontsize=11, color='#536174')
fig.text(0.085, 0.02,
         f"{export['export_seconds']:.1f} s export · {export['billable_seconds']} s billable · FP32 exhaustive search · top 800 per objective\n"
         'Each point covers approximately 3,200 sessions. These are batch export timings, not serving latency or recommendation quality.',
         fontsize=9, color='#536174')
fig.subplots_adjust(left=0.085, right=0.98, top=0.78, bottom=0.23)
fig.savefig(ROOT / 'reports/figures/two_tower_export.png', dpi=180, facecolor=fig.get_facecolor())
plt.show()

## Paired retrieval result

The frozen baseline is recomputed for the **same 103,468 sessions**. It combines revisit/co-visitation sources (up to 1,200 candidates per source) with Item2Vec top 800. The neural source is added at six depths; the resulting union has a larger budget.

An *ideal top-20 ceiling* caps recoverable hits and the ground-truth denominator at 20 per session/objective. It measures what a perfect ranker could recover from a pool. Only the neural list at depth 20 is an actual ordered Recall@20. The union's 74.18% ceiling is not a final ranked score.

The weighting is 0.10 clicks + 0.30 carts + 0.60 orders, following the [official OTTO evaluation definition](https://github.com/otto-de/recsys-dataset/blob/main/KAGGLE.md).

In [ ]:
result = json.loads((ROOT / 'reports/metrics/two_tower_fold0_retrieval.json').read_text())
audit = json.loads((ROOT / 'reports/metrics/two_tower_fold0_audit.json').read_text())
assert result['status'] == audit['status'] == 'passed'
assert result['input_id'] == audit['input_id']
points = result['points']
frontier = pd.DataFrame([{
    'Neural depth': row['neural_k'],
    'Neural pool ceiling (%)': 100 * row['weighted_neural_ceiling'],
    'Base + neural ceiling (%)': 100 * row['weighted_union_ceiling'],
    'Incremental gain (pp)': 100 * row['weighted_incremental_ceiling'],
    'Paired 95% interval (pp)': ' – '.join(f'{100 * x:.3f}' for x in row['weighted_incremental_ci95']),
} for row in points])
print(f"Neural ordered Recall@20: {100 * points[0]['weighted_neural_ceiling']:.3f}%")
print(f"Frozen baseline candidate ceiling: {100 * points[-1]['weighted_base_ceiling']:.3f}%")
display(frontier.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6.1), facecolor='#f7f9fc',
                         gridspec_kw={'width_ratios': [1.3, 1]})
for ax in axes:
    ax.set_facecolor('#f7f9fc')
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color('#b7c3d0')
x = np.arange(len(points))
gain = np.array([row['weighted_incremental_ceiling'] for row in points]) * 100
intervals = np.array([row['weighted_incremental_ci95'] for row in points]) * 100
axes[0].fill_between(x, intervals[:, 0], intervals[:, 1], color='#d2e3f5')
axes[0].errorbar(x, gain, yerr=np.stack([gain - intervals[:, 0], intervals[:, 1] - gain]),
                 fmt='o-', color='#3366b0', lw=2.4, capsize=4, markersize=6)
axes[0].set_xticks(x, [str(row['neural_k']) for row in points])
axes[0].set_ylim(0, 1.25)
axes[0].set_xlabel('Added neural candidates per objective')
axes[0].set_ylabel('Weighted ceiling gain (percentage points)')
axes[0].set_title('Coverage grows with candidate depth', loc='left', fontsize=12, pad=14)
axes[0].grid(axis='y', color='#dce3ec', linewidth=0.8)
axes[0].annotate(f'+{gain[-1]:.3f} pp', xy=(5, gain[-1]), xytext=(4.2, 1.18),
                 color='#17324f', fontweight='bold', fontsize=11)
last = points[-1]
objectives = ['clicks', 'carts', 'orders']
colors = ['#3366b0', '#168777', '#b77a2b']
y = np.arange(3)
values = np.array([last['objectives'][o]['incremental_ceiling'] for o in objectives]) * 100
ci = np.array([last['objectives'][o]['incremental_ci95'] for o in objectives]) * 100
axes[1].barh(y, values, height=0.48, color=colors, alpha=0.9)
axes[1].errorbar(values, y, xerr=np.stack([values - ci[:, 0], ci[:, 1] - values]),
                 fmt='none', ecolor='#17324f', capsize=5, lw=1.3)
for pos, val in zip(y, values):
    axes[1].text(val + 0.17, pos, f'+{val:.2f}', va='center', fontsize=11, color='#17324f')
axes[1].set_yticks(y, [o.title() for o in objectives])
axes[1].invert_yaxis()
axes[1].set_xlim(0, 3)
axes[1].set_xlabel('Objective ceiling gain (percentage points)')
axes[1].set_title('Each objective gains at depth 800', loc='left', fontsize=12, pad=14)
axes[1].grid(axis='x', color='#dce3ec', linewidth=0.8)
fig.suptitle('OTTO | Neural retrieval adds complementary coverage', x=0.085, ha='left',
             fontsize=18, fontweight='bold', color='#17324f')
fig.text(0.085, 0.875,
         f"Fold 0 · {result['sessions']:,} sessions · frozen base {last['weighted_base_ceiling']:.2%} → union {last['weighted_union_ceiling']:.2%}",
         fontsize=11, color='#536174')
fig.text(0.085, 0.065, 'Intervals: 500 paired session bootstrap draws. Fold 0 was used for checkpoint selection.',
         fontsize=9, color='#536174')
fig.text(0.085, 0.032, 'Candidate ceilings are not ranked scores. Notebook 08 evaluates the separate 100-candidate baseline ranker.',
         fontsize=9, color='#536174')
fig.subplots_adjust(left=0.085, right=0.96, bottom=0.23, top=0.76, wspace=0.36)
fig.savefig(ROOT / 'reports/figures/two_tower_retrieval.png', dpi=180, facecolor=fig.get_facecolor())
plt.show()

### Where the additional positives come from

The top-800 neural source recovers 2,385 click, 430 cart, and 138 order positive items that the base pool missed. These are session/objective/item hits, not unique catalogue items or unique users. Raw exclusive hits and capped incremental recall are distinct quantities; adding a positive after a pool already covers 20 cannot improve its top-20 ceiling.

In [ ]:
display(pd.DataFrame([{
    'Objective': o.title(),
    'Capped ground-truth denominator': last['objectives'][o]['denominator'],
    'Base ceiling (%)': 100 * last['objectives'][o]['base_ceiling'],
    'Union ceiling (%)': 100 * last['objectives'][o]['union_ceiling'],
    'Neural-only positive hits': last['objectives'][o]['neural_only_positive_hits'],
    'Gain (pp)': 100 * last['objectives'][o]['incremental_ceiling'],
} for o in objectives]).round(3))

## Independent audit and durable recovery

All 32 count files and receipts were downloaded from the completed S3 comparison. The auditor checks file hashes, immutable identity, session uniqueness and partition membership, count bounds and monotonicity, and the complete cohort size. It reproduces all point estimates and all 500 paired-bootstrap intervals using sample multiplicities, independently of the evaluator's row-gather summary. It checks aggregation of saved counts; it does not rerun raw-label retrieval.

The comparison has UTC JSONL logs, progress/heartbeats, per-part timings, and total attempt timing in S3. Resuming the same input contract verifies completed parts before reuse. The audit reads those small checkpoints; it never loads the multi-gigabyte baseline or trained weights. Interrupted audits can restart from these existing parts. Its final report is written atomically, and the published audit log is in `reports/logs/two_tower_fold0_audit.jsonl`.

In [ ]:
display(pd.DataFrame([
    ('Comparison identity', result['input_id']),
    ('Verified count parts', audit['verified_parts']),
    ('Verified held-out sessions', audit['sessions']),
    ('Independently verified bootstrap draws', audit['bootstrap_iterations_verified']),
    ('Absolute numerical tolerance', audit['absolute_tolerance']),
    ('Comparison attempt seconds', result['elapsed_seconds_this_attempt']),
    ('Retained bucket compute seconds', round(result['completed_bucket_compute_seconds'], 3)),
    ('Independent audit seconds', audit['elapsed_seconds']),
    ('Audit completed (UTC)', audit['verified_at_utc']),
], columns=['Evidence', 'Observed']))

## Decision and next experiment

**Retain the two-tower retriever as a prospective complementary source.** At depth 800, its weighted incremental ceiling is +1.029 pp with a paired 95% interval of +0.931 to +1.120 pp. The standalone ordered Recall@20 is 21.867%; these results do not justify replacing the established retrieval system.

**Completed follow-ups:** [notebook 06](06_ann_benchmark.ipynb) measures ANN fidelity, search cost and the frozen-baseline comparison. [Notebook 08](08_ranking_evaluation.ipynb) measures the separate 30-feature, 100-candidate ranker baseline. That ranker does not yet include certified neural-source features, and its compressed-pool ceiling is not the larger union ceiling reported here.

Next, compare candidate budgets and source coverage alongside broad feature discovery, training-only screening and inner-validated feature-group ablations. A valid base-only versus base-plus-neural ranker comparison requires retriever fitting and checkpoint selection that exclude the relevant evaluation labels. Preserve an untouched temporal evaluation for model-selection-independent evidence. No additional model folds are started by publishing this notebook.

The intervals reflect session resampling for a fixed selected checkpoint. They exclude checkpoint/model selection, retraining variation, and later temporal shifts. No leaderboard or state-of-the-art claim follows from this single-fold experiment; full-test prediction and Kaggle submission remain separate milestones.
